# PSEO Employment Flow Extraction (Agg 178)

Pre-processes `pseof_all.csv` to produce a filtered, analysis-ready flow dataset for a three-tier Sankey visualization (State → Major → Industry × Census Division).

Filters applied:
- `agg_level_pseo == 178` (State × Major × Cohort × Industry × Geography)
- `degree_level == '05'` (Bachelor's)
- `cipcode` in `{'11', '13', '51'}` (Computer Science, Education, Health Professions)

Reads in chunks to avoid loading the full 3 GB / 6M-row file into memory.

In [11]:
import pandas as pd

INPUT_FILE  = "pseof_all.csv"
CHUNK_SIZE  = 50_000

TARGET_AGG_LEVEL   = 178
TARGET_DEGREE_LEVEL = "05"

CIP_SET = {"11", "13", "51"}

OUTPUT_FILE_RAW   = "pseof_agg178_deg5_cip11_13_51.csv"
OUTPUT_FILE_CLEAN = "pseof_agg178_deg5_cip11_13_51_clean.csv"

print(f"Reading '{INPUT_FILE}' in chunks of {CHUNK_SIZE:,} rows...")

Reading 'pseof_all.csv' in chunks of 50,000 rows...


## Step 1 — Chunk Extraction

Streams through the raw file in 50,000-row chunks, keeping only rows that match all three filter conditions. The filtered rows are concatenated and saved to a raw intermediate CSV.

In [12]:
dtype_settings = {
    "agg_level_pseo": int,
    "institution":    str,
    "cipcode":        str,
    "degree_level":   str,
    "grad_cohort":    str,
    "geography":      str,
    "industry":       str,
}


def _normalize_cip(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.zfill(2)


chunks: list[pd.DataFrame] = []

for i, chunk in enumerate(
    pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE, dtype=dtype_settings, low_memory=False)
):
    chunk["cipcode"]     = _normalize_cip(chunk["cipcode"])
    chunk["degree_level"] = chunk["degree_level"].astype(str).str.strip().str.zfill(2)

    filtered = chunk[
        (chunk["agg_level_pseo"] == TARGET_AGG_LEVEL)
        & (chunk["degree_level"] == TARGET_DEGREE_LEVEL)
        & (chunk["cipcode"].isin(CIP_SET))
    ]

    if not filtered.empty:
        chunks.append(filtered)

    if (i + 1) % 10 == 0:
        print(f"  Scanned ~{(i + 1) * CHUNK_SIZE:,} rows so far...")

if chunks:
    df = pd.concat(chunks, ignore_index=True)
    df.to_csv(OUTPUT_FILE_RAW, index=False)
    print(f"\nSaved {len(df):,} rows to '{OUTPUT_FILE_RAW}'")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
else:
    print("No matching rows found. Double-check filters.")

  Scanned ~500,000 rows so far...
  Scanned ~1,000,000 rows so far...
  Scanned ~1,500,000 rows so far...
  Scanned ~2,000,000 rows so far...
  Scanned ~2,500,000 rows so far...
  Scanned ~3,000,000 rows so far...
  Scanned ~3,500,000 rows so far...
  Scanned ~4,000,000 rows so far...
  Scanned ~4,500,000 rows so far...
  Scanned ~5,000,000 rows so far...
  Scanned ~5,500,000 rows so far...
  Scanned ~6,000,000 rows so far...
  Scanned ~6,500,000 rows so far...
  Scanned ~7,000,000 rows so far...
  Scanned ~7,500,000 rows so far...
  Scanned ~8,000,000 rows so far...
  Scanned ~8,500,000 rows so far...
  Scanned ~9,000,000 rows so far...
  Scanned ~9,500,000 rows so far...
  Scanned ~10,000,000 rows so far...
  Scanned ~10,500,000 rows so far...
  Scanned ~11,000,000 rows so far...
  Scanned ~11,500,000 rows so far...
  Scanned ~12,000,000 rows so far...
  Scanned ~12,500,000 rows so far...
  Scanned ~13,000,000 rows so far...
  Scanned ~13,500,000 rows so far...
  Scanned ~14,000,000 

## Step 2 — Sanity Check

In [13]:
df = pd.read_csv(OUTPUT_FILE_RAW, dtype={"cipcode": str, "degree_level": str, "geography": str, "industry": str})

print("=" * 80)
print(f"File : {OUTPUT_FILE_RAW}")
print(f"Shape: {df.shape}")
print(f"\nUnique agg_level_pseo : {sorted(df['agg_level_pseo'].dropna().unique().tolist())}")
print(f"Unique degree_level   : {sorted(df['degree_level'].unique().tolist())}")
print(f"\nRow counts by cipcode:")
print(df["cipcode"].value_counts().sort_index())
print(f"\nRow counts by geography (Census Division):")
print(df["geography"].value_counts().sort_index())
print(f"\nRow counts by industry (NAICS Sector):")
print(df["industry"].value_counts().sort_index())

File : pseof_agg178_deg5_cip11_13_51.csv
Shape: (113040, 30)

Unique agg_level_pseo : [178]
Unique degree_level   : ['05']

Row counts by cipcode:
cipcode
11    38160
13    37080
51    37800
Name: count, dtype: int64

Row counts by geography (Census Division):
geography
1    12560
2    12560
3    12560
4    12560
5    12560
6    12560
7    12560
8    12560
9    12560
Name: count, dtype: int64

Row counts by industry (NAICS Sector):
industry
11       5652
21       5652
22       5652
23       5652
31-33    5652
42       5652
44-45    5652
48-49    5652
51       5652
52       5652
53       5652
54       5652
55       5652
56       5652
61       5652
62       5652
71       5652
72       5652
81       5652
92       5652
Name: count, dtype: int64


## Step 3 — Suppression Handling & Data Nullification

The PSEO flow file uses status flags to indicate data quality:

| Status | Meaning |
|--------|---------|
| `1`    | Valid and released |
| `5`    | Suppressed (fewer than 30 graduates — protected count) |
| `-1`   | Not applicable for this aggregation level |

**Employment value columns** (`y1_grads_emp`, `y1_grads_emp_instate`, `y5_grads_emp`, `y5_grads_emp_instate`, `y10_grads_emp`, `y10_grads_emp_instate`) and **NME columns** (`y1_grads_nme`, `y5_grads_nme`, `y10_grads_nme`) are nulled when:
- The corresponding status flag is `5` (suppressed), or
- The raw value is ≤ 0 (zero or negative counts are non-informative)

**Row deletion rule**: A row is dropped only if **all** employment status flags are suppressed (`5`). Rows with at least one valid employment column are retained — their invalid columns are nulled so they are excluded from aggregations automatically.

The NME columns are retained as-is after nullification because they represent graduates with no or marginal employment and are required to compute the total graduate population denominator.

In [18]:
# All status columns and their corresponding value columns
STATUS_TO_VALUE = {
    "status_y1_grads_emp":          "y1_grads_emp",
    "status_y1_grads_emp_instate":  "y1_grads_emp_instate",
    "status_y5_grads_emp":          "y5_grads_emp",
    "status_y5_grads_emp_instate":  "y5_grads_emp_instate",
    "status_y10_grads_emp":         "y10_grads_emp",
    "status_y10_grads_emp_instate": "y10_grads_emp_instate",
    "status_y1_grads_nme":          "y1_grads_nme",
    "status_y5_grads_nme":          "y5_grads_nme",
    "status_y10_grads_nme":         "y10_grads_nme",
}

# Only employment columns (not NME) are used for the row-deletion decision
EMP_STATUS_COLS = [
    "status_y1_grads_emp",
    "status_y1_grads_emp_instate",
    "status_y5_grads_emp",
    "status_y5_grads_emp_instate",
    "status_y10_grads_emp",
    "status_y10_grads_emp_instate",
]

SUPPRESSED = 5

df_clean = df.copy()
rows_before = len(df_clean)

# --- Row deletion: drop only if ALL employment status flags are suppressed ---
emp_valid = df_clean[EMP_STATUS_COLS].apply(
    lambda col: col.astype(str).str.strip().astype(float) != SUPPRESSED
)
has_any_valid = emp_valid.any(axis=1)
df_clean = df_clean[has_any_valid].reset_index(drop=True)
rows_after = len(df_clean)

print(f"Rows before deletion : {rows_before:,}")
print(f"Rows after  deletion : {rows_after:,}")
print(f"Rows dropped         : {rows_before - rows_after:,}")

# --- Nullify individual value columns where status == 5 or value <= 0 ---
nulled_cells = 0
for status_col, value_col in STATUS_TO_VALUE.items():
    if value_col not in df_clean.columns:
        continue
    status_vals = df_clean[status_col].astype(str).str.strip().astype(float)
    value_vals  = pd.to_numeric(df_clean[value_col], errors="coerce")

    mask_suppress = status_vals == SUPPRESSED
    # mask_nonpos   = value_vals <= 0
    mask_null     = mask_suppress

    df_clean.loc[mask_null, value_col] = float("nan")
    nulled_cells += mask_null.sum()

print(f"Value cells nulled (suppressed or ≤0) : {nulled_cells:,}")

df_clean.to_csv(OUTPUT_FILE_CLEAN, index=False)
print(f"\nSaved cleaned dataset to '{OUTPUT_FILE_CLEAN}'")

Rows before deletion : 113,040
Rows after  deletion : 108,900
Rows dropped         : 4,140
Value cells nulled (suppressed or ≤0) : 269,280

Saved cleaned dataset to 'pseof_agg178_deg5_cip11_13_51_clean.csv'


## Step 4 — Label Mapping

Replaces numeric codes with human-readable labels:

- **`institution`** → State name from `label_fipsnum.csv` (zero-padded FIPS match)
- **`industry`** → NAICS Sector title from `label_industry.csv` (sector-level only, `ind_level == 'S'`)
- **`geography`** → Census Division name from `label_geography_division.csv` (division-level only, `geo_level == 'D'`)
- **`cipcode`** → Major name (hard-coded for the three target CIPs)

Original numeric code columns are retained alongside the new label columns for traceability.

In [19]:
df_clean = pd.read_csv(
    OUTPUT_FILE_CLEAN,
    dtype={"cipcode": str, "degree_level": str, "geography": str, "industry": str},
)

# --- State name labels (FIPS → state name) ---
fips = pd.read_csv("label_fipsnum.csv", dtype={"geography": str})
fips = fips.rename(columns={"label": "state_name"})[["geography", "state_name"]]
df_clean["institution_fips"] = df_clean["institution"].astype(str).str.zfill(2)
df_clean = df_clean.merge(fips, left_on="institution_fips", right_on="geography", how="left")
df_clean = df_clean.drop(columns=["institution_fips", "geography_y"], errors="ignore")
if "geography_x" in df_clean.columns:
    df_clean = df_clean.rename(columns={"geography_x": "geography"})

# --- Industry labels (NAICS Sector level only) ---
ind_labels = pd.read_csv("label_industry.csv", dtype={"industry": str})
ind_labels = ind_labels[ind_labels["ind_level"] == "S"][["industry", "label"]]
ind_labels = ind_labels.rename(columns={"label": "industry_label"})

df_clean = df_clean.merge(ind_labels, on="industry", how="left")

# --- Geography labels (Census Division level only) ---
geo_labels = pd.read_csv("label_geography_division.csv", dtype={"geography": str})
geo_labels = geo_labels[["geography", "label"]].rename(columns={"label": "division_label"})

df_clean = df_clean.merge(geo_labels, on="geography", how="left")

# --- CIP major labels ---
CIP_LABEL = {"11": "Computer Science", "13": "Education", "51": "Health Professions"}
df_clean["major_label"] = df_clean["cipcode"].map(CIP_LABEL)

# Reorder: put label columns immediately after their code columns
cols = df_clean.columns.tolist()
for label_col, ref_col in [
    ("state_name", "institution"),
    ("major_label", "cipcode"),
    ("industry_label", "industry"),
    ("division_label", "geography"),
]:
    if label_col in cols:
        cols.remove(label_col)
        cols.insert(cols.index(ref_col) + 1, label_col)
df_clean = df_clean[cols]

df_clean.to_csv(OUTPUT_FILE_CLEAN, index=False)

print(f"Rows : {len(df_clean):,}")
print(f"Cols : {df_clean.columns.tolist()}")
print(f"\nUnmatched state_names : {df_clean['state_name'].isna().sum()}")
print(f"Unmatched industries  : {df_clean['industry_label'].isna().sum()}")
print(f"Unmatched divisions   : {df_clean['division_label'].isna().sum()}")
print(f"Unmatched majors      : {df_clean['major_label'].isna().sum()}")
print(f"\nSaved labelled dataset to '{OUTPUT_FILE_CLEAN}'")


Rows : 108,900
Cols : ['agg_level_pseo', 'inst_level', 'institution', 'state_name', 'degree_level', 'cip_level', 'cipcode', 'major_label', 'grad_cohort', 'grad_cohort_years', 'geo_level', 'geography', 'division_label', 'ind_level', 'industry', 'industry_label', 'y1_grads_emp', 'y1_grads_emp_instate', 'y5_grads_emp', 'y5_grads_emp_instate', 'y10_grads_emp', 'y10_grads_emp_instate', 'y1_grads_nme', 'y5_grads_nme', 'y10_grads_nme', 'status_y1_grads_emp', 'status_y1_grads_emp_instate', 'status_y5_grads_emp', 'status_y5_grads_emp_instate', 'status_y10_grads_emp', 'status_y10_grads_emp_instate', 'status_y1_grads_nme', 'status_y5_grads_nme', 'status_y10_grads_nme']

Unmatched state_names : 0
Unmatched industries  : 0
Unmatched divisions   : 0
Unmatched majors      : 0

Saved labelled dataset to 'pseof_agg178_deg5_cip11_13_51_clean.csv'


## Step 5 — Cohort Label

Each `grad_cohort` value is the starting year of a 3-year window (e.g., `2001` → graduates of 2001–2003). A `grad_cohort_label` string column (e.g., `"2001-2003"`) is added for use in visualizations.

In [20]:
df_clean = pd.read_csv(
    OUTPUT_FILE_CLEAN,
    dtype={"cipcode": str, "degree_level": str, "geography": str, "industry": str},
)

df_clean["grad_cohort"] = pd.to_numeric(df_clean["grad_cohort"], errors="coerce")
df_clean["grad_cohort_label"] = df_clean["grad_cohort"].apply(
    lambda y: f"{int(y)}-{int(y) + 2}" if pd.notna(y) else None
)

df_clean.to_csv(OUTPUT_FILE_CLEAN, index=False)
print(f"Columns: {df_clean.columns.tolist()}")
print(f"\nSample cohort labels:")
print(df_clean[["grad_cohort", "grad_cohort_label"]].drop_duplicates().sort_values("grad_cohort").to_string(index=False))

Columns: ['agg_level_pseo', 'inst_level', 'institution', 'state_name', 'degree_level', 'cip_level', 'cipcode', 'major_label', 'grad_cohort', 'grad_cohort_years', 'geo_level', 'geography', 'division_label', 'ind_level', 'industry', 'industry_label', 'y1_grads_emp', 'y1_grads_emp_instate', 'y5_grads_emp', 'y5_grads_emp_instate', 'y10_grads_emp', 'y10_grads_emp_instate', 'y1_grads_nme', 'y5_grads_nme', 'y10_grads_nme', 'status_y1_grads_emp', 'status_y1_grads_emp_instate', 'status_y5_grads_emp', 'status_y5_grads_emp_instate', 'status_y10_grads_emp', 'status_y10_grads_emp_instate', 'status_y1_grads_nme', 'status_y5_grads_nme', 'status_y10_grads_nme', 'grad_cohort_label']

Sample cohort labels:
 grad_cohort grad_cohort_label
        2001         2001-2003
        2004         2004-2006
        2007         2007-2009
        2010         2010-2012
        2011         2011-2013
        2013         2013-2015
        2016         2016-2018
        2019         2019-2021


# Double Snakey With Labels for Home State with the State Name
### Also to allow toggle between industry only, state only, and industry and state both.